# Experiment 2: Metadata 
This second experiment consists on seeing which is the best technique concerning metadata: 
- No using metadata
- Using metadata and concatenate it to the text embeddings 
- Using metadata with it's separate embeddings and have a score on both embeddings

In [101]:
import os, json
import google.generativeai as genai
import uuid
import fitz  #pip install pymupdf
import json
import tempfile
import requests
import numpy as np
from pathlib import Path
import sys
import requests

from embedder import Embedder

In [102]:
# we are in: backend/exp2/experiment_2_metadata.ipynb
BASE_DIR = Path.cwd().parents[0]   # backend/
DATA_DIR = BASE_DIR / "data"
PDF_DIR = DATA_DIR / "pdfs"
PDF_EXAMPLES = BASE_DIR/ "example_pdfs_to_upload"

sys.path.append(str(BASE_DIR))

'''print("PDF_DIR:", PDF_DIR)
print("PDFs:", len(list(PDF_DIR.glob("*.pdf"))))
print("DB exists:", DB_PATH.exists())'''
print("PDF_DIR:", PDF_EXAMPLES)
print("PDFs:", len(list(PDF_EXAMPLES.glob("*.pdf"))))



PDF_DIR: c:\Lucía\Lucia\uni\z otras cosas\ERASMUS\VIENA\asignaturas\GenAI\GenAI-PR-2025w\backend\example_pdfs_to_upload
PDFs: 28


In [ ]:
#gemini metadata 

#EVERYONE NEEDS THEIR API KEY
#environment variable
#Windows powershell => setx GEMINI_API_KEY "YOUR_API_KEY"
#Linux/MacOs => export GEMINI_API_KEY="YOUR_API_KEY"

os.environ["GEMINI_API_KEY"] = "YOUR API KEY"
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY env var")


genai.configure(api_key=api_key)

#GEMINI_MODEL = "gemini-2.0-flash" 
GEMINI_MODEL = "gemini-2.0-flash-lite"

META_CACHE_PATH = DATA_DIR / "llm_metadata_cache.json"

if META_CACHE_PATH.exists():
    metadata_cache = json.loads(META_CACHE_PATH.read_text(encoding="utf-8"))
else:
    metadata_cache = {}


def extract_llm_metadata(doc_text):
    """
    Returns metadata in JSON from the text of the document
    """

    #client = genai.Client(api_key=api_key)

    # recorta para no pasarle el documento entero (suficiente con inicio/abstract)
    snippet = doc_text[:12000]

    prompt = f"""
        You are extracting bibliographic and topical metadata from a PDF text dump.
        Return ONLY valid JSON (no markdown).

        Schema:
        {{
        "title": string|null,
        "authors": [string],
        "year": int|null,
        "keywords": [string],   // 8-15 items
        "topics": [string],     // 2-5 short tags
        "one_sentence_summary": string|null
        }}

        Rules:
        - If you are unsure, use null or empty lists.
        - Keep keywords/topics concise (1-4 words).
        - Do not hallucinate specific author names if not present.
        - Base everything only on the provided text.

        TEXT:
        {snippet}
        """.strip()

    url = (
        "https://generativelanguage.googleapis.com/v1beta/"
        "models/gemini-2.5-flash:generateContent"
        )

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": api_key
    }

    payload = {
        "contents": [
            {"parts": [{"text": prompt}]}
        ]
    }

    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    data = response.json()

    raw = data["candidates"][0]["content"]["parts"][0]["text"]

    # quitar ```json y ```
    raw = raw.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print("\n=== JSON PARSE FAILED ===")
        print("Error:", e)
        print("RAW OUTPUT:\n", raw)
        print("========================\n")
        raise
    #text = data["candidates"][0]["content"]["parts"][0]["text"]
    
    return json.loads(raw)


import time
def get_llm_metadata(pdf_name, doc_text):
    if pdf_name not in metadata_cache:
        time.sleep(4)
        metadata_cache[pdf_name] = extract_llm_metadata(doc_text)
        META_CACHE_PATH.write_text(json.dumps(metadata_cache, indent=2), encoding="utf-8")
    return metadata_cache[pdf_name]

def build_metadata_string(llm_meta):

    if not llm_meta:
        return ""
    parts = []
    if llm_meta.get("title"):
        parts.append(f"Title: {llm_meta['title']}")
    if llm_meta.get("topics"):
        parts.append("Topics: " + ", ".join(llm_meta["topics"]))
    if llm_meta.get("keywords"):
        parts.append("Keywords: " + ", ".join(llm_meta["keywords"]))
    if llm_meta.get("one_sentence_summary"):
        parts.append("Summary: " + llm_meta["one_sentence_summary"])
    return "[METADATA]\n" + "\n".join(parts)

In [ ]:
# data_manager.py

class DataManager:
    """
    Handles everything related to:
    - downloading or receiving PDFs
    - saving them to the database
    - extracting text
    - chunking, embedding & storing database
    """

    def __init__(self, pdf_folder=PDF_EXAMPLES, metadata_mode = "concat"): #PDF_DIR
        self.embedder = Embedder()
        self.pdf_folder = pdf_folder
        self.metadata_mode = metadata_mode #"concat" or "dual"
        self.tokenizer = self.embedder.model.tokenizer


    # ------------------------------
    # PROCESSING (CHUNK + EMBEDDING)
    # ------------------------------
    def extract_text(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = "".join([page.get_text() for page in doc])
        doc.close()
        return text

    def chunk_text(self, text, chunk_size=400, chunk_overlap=50):
        # 1. Convert text to token IDs (integers)
        # We use add_special_tokens=False so we don't get [CLS]/[SEP] inside every chunk
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        chunks = []
        # 2. Iterate through tokens with a sliding window
        # The step size is (chunk_size - chunk_overlap)
        step = chunk_size - chunk_overlap
        for i in range(0, len(tokens), step):
            # Extract the window of tokens
            chunk_ids = tokens[i : i + chunk_size]
            # 3. Decode back to text
            chunk_text = self.tokenizer.decode(chunk_ids, skip_special_tokens=True)
            chunks.append(chunk_text)
        return chunks

    '''def process_pdf(self, entry):
        pdf_path = os.path.join(self.pdf_folder, entry["pdf_name"])
        if not os.path.exists(pdf_path):
            print("PDF missing:", pdf_path)
            return'''


    def process_pdf(self, pdf_path, mode = "baseline"):
        '''pdf_path = os.path.join(self.pdf_folder, entry["pdf_name"])
        if not os.path.exists(pdf_path):
            print("PDF missing:", pdf_path)
            return'''

        text = self.extract_text(pdf_path)
        chunks = self.chunk_text(text)

        llm_meta = {}
        meta_str = ""
        if mode in ("concat", "dual"):
            llm_meta = get_llm_metadata(pdf_path.name, text) or {}
            meta_str = build_metadata_string(llm_meta)
    
        #metadata doc-level
        '''self.ensure_llm_metadata(entry, text)
        metadata_str = self.build_metadata_string(entry)
        '''
        if mode == "dual": 
            emb_meta = self.embedder.encode(meta_str)

        chunk_records = []
        for chunk in chunks: 
            if mode == "baseline":
                emb = self.embedder.encode(chunk)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "concat":
                emb = self.embedder.encode(chunk + "\n\n" + meta_str)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "dual":
                emb_text = self.embedder.encode(chunk)
                chunk_records.append({"text": chunk, "embedding_text": emb_text, "embedding_meta": emb_meta})
        
        return {"pdf_name": pdf_path.name, "llm_metadata": llm_meta, "chunks": chunk_records}
        '''for chunk in chunks:
            if mode == "concat":
                text_to_embed = chunk + "\n\n" + metadata_str if metadata_str else chunk
                embedding = self.embedder.encode(text_to_embed)

            entry["chunks"].append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "embedding": embedding
            })

        self.save_database()
        print(f"Indexed {len(chunks)} chunks for: {entry['title']}")
        return entry'''

    def build_index(self, pdf_paths, mode = "baseline"):
        self.database = []
        for p in pdf_paths:
            entry = self.process_pdf(p, mode=mode)
            print("PDF processed")
            self.database.append(entry)
        return self.database

In [125]:
#baseline retriever

class BaselineRetriever:
    def __init__(self, database = None, embedder = None, mode = "not_dual"):
        """
        database: list (in-memory database). If provided, we won't load from file.
        database_file: path to JSON, used if database is None.
        embedder: optional shared Embedder instance.
        """
        self.embedder = embedder or Embedder()
        #self.database_file = database_file
        self.database = database #can be None or list
        self.mode = mode #not_dual or dual

    def cosine_similarity(self, v1, v2):
        v1 = np.array(v1)
        v2 = np.array(v2)
        return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

    def search(self, query, threshold=0.70, alpha = 0.7):
        """Returns papers ranked by similarity. threshold ~ 0.65-0.80 recommended"""    

        if len(self.database) == 0:
            print("Database is empty: no entries to search.\n")

        query_emb = self.embedder.encode(query)
        results = []

        for entry in self.database:
            if "chunks" not in entry:
                continue  # not processed yet
            best_score = 0
            best_chunk = None

            for chunk in entry["chunks"]:
                if self.mode == "dual":
                    if chunk.get("embedding_text") is None or chunk.get("embedding_meta") is None:
                        continue
                    s_text = self.cosine_similarity(query_emb, chunk["embedding_text"])
                    s_meta = self.cosine_similarity(query_emb, chunk["embedding_meta"])

                    score = alpha * s_text + (1 - alpha) * s_meta
                else:
                    score = self.cosine_similarity(query_emb, chunk["embedding"])
                
                if score > best_score:
                    best_score = score
                    best_chunk = chunk
            
            if best_score >= threshold:
                results.append({
                "paper_id": entry.get("id", entry.get("pdf_name", "")),
                "title": (entry.get("llm_metadata") or {}).get("title", entry.get("pdf_name", "")), #"title": entry.get("llm_metadata", {}).get("title", entry.get("pdf_name")),
                "pdf_name": entry.get("pdf_name", ""),
                "score": round(best_score, 3),
                "sample_text": best_chunk["text"][:300] if best_chunk else ""
})
        # Sort best match → worst
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

In [113]:
embedder = Embedder()

#dm = DataManager(embedder = embedder, max_chars = 1000)
dm = DataManager()

#pdf_paths = list(PDF_DIR.glob("*.pdf"))
pdf_paths = list(PDF_EXAMPLES.glob("*.pdf"))
import random
random.seed(0)
subset_paths = random.sample(pdf_paths, k=10)

print("PDFs to index:", len(subset_paths))

PDFs to index: 10


In [114]:
baseline_db = dm.build_index(subset_paths, mode="baseline")

Token indices sequence length is longer than the specified maximum sequence length for this model (734 > 256). Running this sequence through the model will result in indexing errors


PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [115]:
baseline_db[0]["llm_metadata"]

{}

In [116]:
concat_db = dm.build_index(subset_paths, mode="concat")

PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [117]:
concat_db[0]["llm_metadata"]

{'title': 'Meeting Summary: Tau-Protein Data Gap and Model Progress',
 'authors': ['Daniel Fischer'],
 'year': 2026,
 'keywords': ['Tau-protein data gap',
  'AD-Fusion-v2 pipeline',
  'Data quality issues',
  'PET-Tau tracers',
  'Data auditing',
  'ETL scripts',
  'Latent diffusion training',
  'Transformer architecture',
  'Dynamic masking',
  'Cross-attention layer',
  'Multimodal inputs',
  'H100 cluster',
  'OASIS-3 data standard',
  'Explainable AI',
  'Generative models'],
 'topics': ['Neuroscience data',
  'Machine learning models',
  'Data quality management',
  'AI development workflow',
  'Biomedical imaging'],
 'one_sentence_summary': 'This meeting addressed a critical Tau-protein data gap affecting the AD-Fusion-v2 pipeline, necessitating an audit, while concurrently developing architectural adaptations using dynamic masking to manage incomplete multimodal inputs for the generative model.'}

In [118]:
concat_db[1]["llm_metadata"]

{'title': 'Multimodal Attention-based Deep Learning for Alzheimer’s Disease Diagnosis',
 'authors': ['Michal Golovanevsky', 'Carsten Eickhoff', 'Ritambhara Singh'],
 'year': 2022,
 'keywords': ["Alzheimer's Disease Diagnosis",
  'Deep Learning',
  'Multimodal Attention',
  'Mild Cognitive Impairment',
  'Cross-modal Attention',
  'Clinical Decision Support',
  'Medical Imaging',
  'Genetic Data',
  'Clinical Data',
  'Machine Learning',
  'Neurodegenerative Disorders',
  'Disease Classification',
  'ADNI Dataset',
  'Model Performance',
  'Attention Mechanisms'],
 'topics': ['Medical AI',
  "Alzheimer's Research",
  'Multimodal Diagnostics',
  'Deep Learning Applications'],
 'one_sentence_summary': 'This study develops a novel multimodal deep learning framework (MADDi) leveraging cross-modal attention to integrate imaging, genetic, and clinical data for highly accurate multi-class classification of Alzheimer’s Disease, mild cognitive impairment, and controls.'}

In [119]:
dual_db = dm.build_index(subset_paths, mode="dual")

PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [126]:
baseline_retriever = BaselineRetriever(database = baseline_db, embedder = embedder)
#concat_retriever   = BaselineRetriever(concat_db, embedder)
#dual_retriever     = BaselineRetriever(dual_db, embedder) 

In [127]:
#baseline_retriever.search("GPT-2 fine-tuning", threshold=0.6)
baseline_retriever.search("scaled dot-product attention", threshold=0.6)

[{'paper_id': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'title': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'pdf_name': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'score': 0.678,
  'sample_text': 'k, v ) = softmax ( qkt √dk ) v ( 1 ) the two most commonly used attention functions are additive attention [ 2 ], and dot - product ( multi - plicative ) attention. dot - product attention is identical to our algorithm, except for the scaling factor of 1 √dk. additive attention computes the compatib'}]

In [128]:
baseline_retriever.search("cows and milk", threshold=0.6)

[]

In [129]:
concat_retriever   = BaselineRetriever(database = concat_db, embedder = embedder)

In [130]:
concat_retriever.search("scaled dot-product attention", threshold=0.6)

[{'paper_id': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'title': 'Attention Is All You Need',
  'pdf_name': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'score': 0.678,
  'sample_text': 'k, v ) = softmax ( qkt √dk ) v ( 1 ) the two most commonly used attention functions are additive attention [ 2 ], and dot - product ( multi - plicative ) attention. dot - product attention is identical to our algorithm, except for the scaling factor of 1 √dk. additive attention computes the compatib'}]

In [131]:
concat_retriever.search("cows and milk", threshold=0.6)

[]

In [132]:
dual_retriever = BaselineRetriever(database = dual_db, embedder = embedder,  mode = "dual")
dual_retriever.search("scaled dot-product attention", threshold=0.6)

[{'paper_id': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'title': 'Attention Is All You Need',
  'pdf_name': '2025-11-10_Chloe-Nguyen_Attention-Is-All-You-Need.pdf',
  'score': 0.619,
  'sample_text': 'k, v ) = softmax ( qkt √dk ) v ( 1 ) the two most commonly used attention functions are additive attention [ 2 ], and dot - product ( multi - plicative ) attention. dot - product attention is identical to our algorithm, except for the scaling factor of 1 √dk. additive attention computes the compatib'}]